# EduVidQA Qwen3.5-4B Vision — Full FT + Full Base-vs-FT Eval

Bản notebook này thay lõi fine-tuning theo hướng Unsloth Qwen3.5 Vision:
- dùng `FastVisionModel` + `UnslothVisionDataCollator`;
- train final trực tiếp bằng config `r16_e1`;
- không còn sweep, không còn fixed50;
- giữ Gemini Round Robin judge 3 keys;
- chạy full eval: **base model vs fine-tuned model** trên `synthetic_test_clean + real_world_test_clean`;
- report rule metrics + judge metrics: `format_pass_rate`, `no_think_rate`, `one_paragraph_rate`, `clarity`, `UPT`, `ECT`, `FactQA Precision/Recall`, `Entailment`, `Groundedness`, `Hallucination`.

Claim hợp lý: đây là experiment dùng EduVidQA như corpus style-alignment cho educational VQA tutor, không phải reproduction đầy đủ benchmark paper.


In [ ]:
# 0) Gemini API keys + experiment config
from getpass import getpass
from google.colab import drive

drive.mount("/content/drive")

# ===== Main switches =====
RUN_FINAL_FULL_TRAIN = True
RUN_FINAL_FULL_EVAL = True
RUN_GEMINI_JUDGE = True

# If FORCE_RERUN=False, notebook resumes from existing jsonl/adapters on Drive.
FORCE_RERUN = False

# ===== Gemini judge: round-robin 3 keys =====
# Full eval calls ~= 2 * (len(synthetic_test) + len(real_world_test)).
# Your previous clean split: 131 + 254 = 385; base+FT = 770 judge calls.
if RUN_GEMINI_JUDGE:
    GEMINI_API_KEY_1 = getpass("Gemini API key #1: ").strip()
    GEMINI_API_KEY_2 = getpass("Gemini API key #2: ").strip()
    GEMINI_API_KEY_3 = getpass("Gemini API key #3: ").strip()
    GEMINI_API_KEYS = [GEMINI_API_KEY_1, GEMINI_API_KEY_2, GEMINI_API_KEY_3]
    assert all(GEMINI_API_KEYS), "Nhap dung 3 Gemini API keys de chay round-robin judge."
    assert len(set(GEMINI_API_KEYS)) == len(GEMINI_API_KEYS), "Ba API keys nen khac nhau de round-robin dung muc tieu."
else:
    GEMINI_API_KEYS = []

GEMINI_MODEL = "gemini-3.1-flash-lite-preview"
GEMINI_RPM_LIMIT_PER_KEY = 15
GEMINI_RPD_LIMIT_PER_KEY = 500
GEMINI_THINKING_LEVEL = "LOW"  # MINIMAL, LOW, MEDIUM, HIGH
GEMINI_RATE_LIMIT_COOLDOWN_SEC = 60

# ===== Model / training =====
BASE_MODEL = "unsloth/Qwen3.5-4B"
MAX_SEQ_LENGTH = 4096

# Unsloth docs warn Qwen3.5 QLoRA/4-bit can have higher quantization differences.
# A100/L4: keep False for bf16/16-bit LoRA if VRAM allows.
# T4 OOM fallback: True, PER_DEVICE_BATCH=1 or 2, GRAD_ACCUM adjusted.
LOAD_IN_4BIT = False

SEED = 42
GEN_MAX_NEW_TOKENS = 256
GEN_TEMPERATURE = 0.1
GEN_TOP_P = 0.9

# r16_e1 final config
SELECTED_CONFIG = {
    "name": "r16_e1",
    "r": 16,
    "alpha": 32,
    "dropout": 0.0,
    "lr": 1e-5,
    "epochs": 1,
    "batch": 4,
    "grad_accum": 2,
    "save_steps": 500,
    "warmup_steps": 10,
    "lr_scheduler_type": "cosine",
    "optim": "adamw_8bit",
}

# ===== Dataset / outputs =====
DATA_ARCHIVE_FILE_ID = "1uXvOVhwo8j944gRYBqpzxKZn0_EqjdzL"
DATA_ARCHIVE_FILENAME = "ft_context_vlm_clean.tar.gz"

DRIVE_WORKDIR = "/content/drive/MyDrive/eduvidqa_qwen35_vl_full_r16e1"
DRIVE_DATA_CACHE = "/content/drive/MyDrive/eduvidqa"
LOCAL_DOWNLOAD_DIR = "/content"

OUTPUT_DIR = f"{DRIVE_WORKDIR}/outputs"
CHECKPOINT_EVERY = 100  # prediction/judge jsonl checkpoint frequency, not trainer save_steps

print("RUN_FINAL_FULL_TRAIN:", RUN_FINAL_FULL_TRAIN)
print("RUN_FINAL_FULL_EVAL:", RUN_FINAL_FULL_EVAL)
print("RUN_GEMINI_JUDGE:", RUN_GEMINI_JUDGE, "keys:", len(GEMINI_API_KEYS))
print("BASE_MODEL:", BASE_MODEL)
print("SELECTED_CONFIG:", SELECTED_CONFIG)
print("OUTPUT_DIR:", OUTPUT_DIR)


In [ ]:
# 1) Install dependencies
# Based on Unsloth Qwen3.5 / Qwen3.5 Vision notebooks.
import importlib.util, os

FORCE_REINSTALL_DEPS = False
INSTALL_QWEN35_FAST_KERNELS = True  # can be slow to compile; set False if install breaks.

def has_module(name):
    return importlib.util.find_spec(name) is not None

need_install = FORCE_REINSTALL_DEPS or not (
    has_module("unsloth")
    and has_module("trl")
    and has_module("transformers")
    and has_module("peft")
    and has_module("google.genai")
    and has_module("gdown")
)

if need_install:
    !pip install -q --upgrade uv

    if FORCE_REINSTALL_DEPS:
        !pip uninstall -y unsloth unsloth_zoo trl transformers peft accelerate bitsandbytes pillow Pillow

    # Official-style Unsloth install for current Colab.
    !uv pip install --system -q --upgrade --no-cache-dir         "unsloth[base] @ git+https://github.com/unslothai/unsloth"         "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo"

    !uv pip install --system -q --upgrade --no-cache-dir         "tokenizers>=0.22.0,<=0.23.0"         "trl==0.22.2"         "transformers==5.2.0"         datasets pandas scikit-learn gdown google-genai Pillow tqdm

    # Helpful for Colab CUDA stack used by Unsloth notebooks.
    !uv pip install --system -q --upgrade --no-cache-dir bitsandbytes xformers==0.0.32.post2 torchvision

    if INSTALL_QWEN35_FAST_KERNELS:
        # If this fails on your runtime, set INSTALL_QWEN35_FAST_KERNELS=False and rerun install.
        !uv pip install --system -q --no-build-isolation flash-linear-attention causal_conv1d==1.6.0 || true
        !uv pip install --system -q --no-deps --upgrade "torchao>=0.16.0" || true

print("Dependency cell finished.")


In [ ]:
# 2) Imports + GPU check
import gc
import inspect
import json
import math
import os
import random
import re
import shutil
import subprocess
import tarfile
import time
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
import torch
from PIL import Image, UnidentifiedImageError
from torch.utils.data import Dataset as TorchDataset
from tqdm.auto import tqdm

torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    subprocess.run(["nvidia-smi"], check=False)


In [ ]:
# 3) Locate/cache/download and extract clean EduVidQA VLM dataset
root = Path("/content/eduvidqa_qwen35_vl_runtime")
root.mkdir(parents=True, exist_ok=True)

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(DRIVE_DATA_CACHE).mkdir(parents=True, exist_ok=True)
Path(LOCAL_DOWNLOAD_DIR).mkdir(parents=True, exist_ok=True)

drive_cache_path = Path(DRIVE_DATA_CACHE) / DATA_ARCHIVE_FILENAME

if not drive_cache_path.exists():
    print("Dataset archive not found on Drive; downloading with gdown...")
    !gdown {DATA_ARCHIVE_FILE_ID} -O {drive_cache_path}

assert drive_cache_path.exists(), f"Dataset archive not found: {drive_cache_path}"

extract_root = root / "dataset"
dataset_root = extract_root / "ft_context_vlm_clean"
summary_path = dataset_root / "summary.json"

if not summary_path.exists():
    print("Extracting dataset to local /content runtime...")
    extract_root.mkdir(parents=True, exist_ok=True)
    with tarfile.open(drive_cache_path, "r:gz") as archive:
        archive.extractall(extract_root)
else:
    print("Dataset already extracted locally.")

print("Dataset archive:", drive_cache_path)
print("Dataset root:", dataset_root)
print("Summary preview:")
print((dataset_root / "summary.json").read_text(encoding="utf-8")[:1200])


In [ ]:
# 4) Load full train/test records

def read_jsonl(path):
    rows = []
    path = Path(path)
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

synthetic_train = read_jsonl(dataset_root / "synthetic_train.jsonl")
synthetic_test = read_jsonl(dataset_root / "synthetic_test.jsonl")
real_world_test = read_jsonl(dataset_root / "real_world_test.jsonl")

full_test_records = synthetic_test + real_world_test

print("synthetic_train:", len(synthetic_train))
print("synthetic_test:", len(synthetic_test))
print("real_world_test:", len(real_world_test))
print("full_test_records:", len(full_test_records))

print("Example keys:", synthetic_train[0].keys())
print("Example id:", synthetic_train[0].get("id"))
print("Example image:", synthetic_train[0].get("image"))
print("Example split:", synthetic_train[0].get("split"))


In [ ]:
# 5) Prompt + lazy vision conversation dataset

SYSTEM_PROMPT = """You are an expert computer science educator answering questions about a lecture video.
Use only the provided frame and transcript window. If the context is insufficient, say so clearly.
Answer as a helpful tutor in one concise paragraph.
Do not use markdown bullets unless absolutely necessary.
Do not reveal hidden reasoning or <think> text."""

USER_TEMPLATE = """{system_prompt}

{record_text_input}

Answer the student question using the frame and transcript. Return only the final answer."""

def image_abs_path(row):
    return str(dataset_root / row["image"])

def make_instruction(row):
    return USER_TEMPLATE.format(
        system_prompt=SYSTEM_PROMPT,
        record_text_input=row["text_input"],
    )

def quick_filter_image_rows(records, split_name):
    good, bad = [], []
    for row in tqdm(records, desc=f"Checking image files ({split_name})", unit="img"):
        path = Path(image_abs_path(row))
        exists = path.exists()
        size = path.stat().st_size if exists else 0
        if exists and size > 1024:
            good.append(row)
        else:
            bad.append({
                "id": row.get("id"),
                "split": row.get("split"),
                "video_id": row.get("video_id"),
                "image": row.get("image"),
                "path": str(path),
                "exists": exists,
                "size": size,
            })

    print(f"{split_name}: good={len(good)} bad={len(bad)}")
    if bad:
        bad_path = Path(OUTPUT_DIR) / f"bad_images_{split_name}.json"
        bad_path.parent.mkdir(parents=True, exist_ok=True)
        bad_path.write_text(json.dumps(bad, ensure_ascii=False, indent=2), encoding="utf-8")
        print("Bad image log:", bad_path)
        print("First bad image:", bad[0])
    return good, bad

class EduVidQALazyConversationDataset(TorchDataset):
    def __init__(self, records, split_name):
        self.records = list(records)
        self.split_name = split_name

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        row = self.records[idx]
        image_path = image_abs_path(row)
        try:
            image = Image.open(image_path).convert("RGB")
        except (UnidentifiedImageError, OSError, ValueError) as exc:
            raise RuntimeError(
                f"Bad image in {self.split_name} at idx={idx}, "
                f"id={row.get('id')}, path={image_path}: {exc}"
            ) from exc

        return {
            "id": row["id"],
            "messages": [
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": image},
                        {"type": "text", "text": make_instruction(row)},
                    ],
                },
                {
                    "role": "assistant",
                    "content": [{"type": "text", "text": row["answer"]}],
                },
            ],
        }

# Full train and full test only.
train_records_for_this_run, bad_train_images = quick_filter_image_rows(synthetic_train, "synthetic_train")
full_test_records, bad_test_images = quick_filter_image_rows(full_test_records, "full_test")

assert train_records_for_this_run, "No valid training images remain after filtering."
assert full_test_records, "No valid test images remain after filtering."

train_dataset = EduVidQALazyConversationDataset(train_records_for_this_run, "synthetic_train")
print("train_dataset:", len(train_dataset))
print("full_test_records:", len(full_test_records))
print("IMPORTANT: this dataset is lazy; it does not call Dataset.from_list and does not preload PIL images.")


In [ ]:
# 6) Model loading, LoRA attach, generation, training helpers
from unsloth import FastVisionModel
from trl import SFTConfig, SFTTrainer
from peft import PeftModel
from unsloth.trainer import UnslothVisionDataCollator

OUTPUT_ROOT = Path(OUTPUT_DIR)
BASELINE_DIR = OUTPUT_ROOT / "baseline_full"
FINAL_DIR = OUTPUT_ROOT / "final_full_r16e1"
for _d in [BASELINE_DIR, FINAL_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

def clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def write_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def write_jsonl(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

def read_jsonl_safe(path):
    path = Path(path)
    if not path.exists():
        return []
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def strip_thinking(text):
    text = text or ""
    text = re.sub(r"<think\b[^>]*>.*?</think>", "", text, flags=re.S | re.I)
    text = re.sub(r"<\|im_end\|>", "", text)
    return text.strip()

def load_base_model():
    print("Loading base model:", BASE_MODEL)
    kwargs = dict(
        model_name=BASE_MODEL,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=LOAD_IN_4BIT,
        fast_inference=False,
        gpu_memory_utilization=0.80,
    )
    try:
        model, processor = FastVisionModel.from_pretrained(**kwargs)
    except TypeError:
        kwargs.pop("gpu_memory_utilization", None)
        model, processor = FastVisionModel.from_pretrained(**kwargs)
    print("Processor:", type(processor))
    print("Tokenizer:", type(processor.tokenizer))
    return model, processor

def attach_lora(model, cfg):
    return FastVisionModel.get_peft_model(
        model,
        finetune_vision_layers=False,       # aim: answer style/format, not visual backbone adaptation
        finetune_language_layers=True,
        finetune_attention_modules=True,
        finetune_mlp_modules=True,
        r=int(cfg["r"]),
        lora_alpha=int(cfg["alpha"]),
        lora_dropout=float(cfg.get("dropout", 0.0)),
        bias="none",
        random_state=SEED,
        use_rslora=False,
        loftq_config=None,
        use_gradient_checkpointing="unsloth",
    )

def load_model_with_adapter(adapter_dir):
    model, processor = load_base_model()
    model = PeftModel.from_pretrained(model, str(adapter_dir))
    return model, processor

def processor_inputs(processor, image, text_prompt):
    # Support both old and new processor call signatures.
    try:
        return processor(
            images=image,
            text=text_prompt,
            add_special_tokens=False,
            return_tensors="pt",
        ).to("cuda")
    except TypeError:
        try:
            return processor(
                text=[text_prompt],
                images=[image],
                videos=[],
                padding=True,
                return_tensors="pt",
            ).to("cuda")
        except TypeError:
            return processor(
                image,
                text_prompt,
                add_special_tokens=False,
                return_tensors="pt",
            ).to("cuda")

def generate_one(model, processor, row, max_new_tokens=GEN_MAX_NEW_TOKENS, temperature=GEN_TEMPERATURE, top_p=GEN_TOP_P):
    image = Image.open(image_abs_path(row)).convert("RGB")
    messages = [{
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": make_instruction(row)},
        ],
    }]

    try:
        text_prompt = processor.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        text_prompt = processor.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

    inputs = processor_inputs(processor, image, text_prompt)
    input_len = inputs["input_ids"].shape[-1]

    t0 = time.perf_counter()
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=temperature,
            top_p=top_p,
            use_cache=True,
        )

    latency = time.perf_counter() - t0
    raw_answer = processor.tokenizer.decode(output_ids[0][input_len:], skip_special_tokens=True)
    clean_answer = strip_thinking(raw_answer)

    return {
        "id": row["id"],
        "split": row.get("split"),
        "video_id": row.get("video_id"),
        "raw_generated_answer": raw_answer,
        "generated_answer": clean_answer,
        "had_think_raw": bool(re.search(r"<think\b", raw_answer or "", flags=re.I)),
        "latency_sec": round(latency, 3),
        "output_tokens": int(output_ids.shape[-1] - input_len),
    }

def generate_predictions(model, processor, rows, out_path, label, force=False):
    out_path = Path(out_path)
    predictions = [] if force else read_jsonl_safe(out_path)
    done = {row["id"] for row in predictions}

    if predictions and len(done) >= len(rows):
        print(f"SKIP generation {label}: {out_path} ({len(predictions)} rows)")
        return predictions

    try:
        FastVisionModel.for_inference(model)
    except Exception as exc:
        print("FastVisionModel.for_inference fallback to model.eval():", repr(exc))

    model.eval()
    for row in tqdm(rows, desc=f"Generate {label}", unit="sample"):
        if row["id"] in done:
            continue
        predictions.append(generate_one(model, processor, row))
        done.add(row["id"])
        if len(predictions) % CHECKPOINT_EVERY == 0:
            write_jsonl(out_path, predictions)

    write_jsonl(out_path, predictions)
    print("saved", len(predictions), out_path)
    return predictions

def _ensure_legacy_eos_token_if_needed(model, processor):
    token = "<EOS_TOKEN>"
    token_id = processor.tokenizer.convert_tokens_to_ids(token)
    if token_id is not None and token_id != processor.tokenizer.unk_token_id:
        return
    added = processor.tokenizer.add_special_tokens({"additional_special_tokens": [token]})
    if added:
        model.resize_token_embeddings(len(processor.tokenizer))
        print(f"Added fallback special token {token!r} for TRL compatibility.")

def train_final_adapter(cfg, train_records, run_dir, force=False):
    run_dir = Path(run_dir)
    adapter_dir = run_dir / "adapter"
    trainer_dir = run_dir / "trainer"
    done_marker = adapter_dir / "adapter_config.json"

    if done_marker.exists() and not force:
        print(f"SKIP training: adapter exists at {adapter_dir}")
        return adapter_dir, None

    clear_cuda()
    model, processor = load_base_model()
    model = attach_lora(model, cfg)
    FastVisionModel.for_training(model)

    train_ds = EduVidQALazyConversationDataset(train_records, "final_train")

    args = SFTConfig(
        per_device_train_batch_size=int(cfg.get("batch", 4)),
        gradient_accumulation_steps=int(cfg.get("grad_accum", 2)),
        num_train_epochs=int(cfg.get("epochs", 1)),
        learning_rate=float(cfg.get("lr", 1e-5)),
        warmup_steps=int(cfg.get("warmup_steps", 10)),
        logging_steps=10,

        # Full final train: no mid-training eval to avoid doubling time.
        eval_strategy="no",

        save_strategy="steps",
        save_steps=int(cfg.get("save_steps", 500)),
        save_total_limit=2,

        optim=str(cfg.get("optim", "adamw_8bit")),
        weight_decay=0.001,
        lr_scheduler_type=str(cfg.get("lr_scheduler_type", "cosine")),
        max_grad_norm=0.3,
        seed=SEED,
        output_dir=str(trainer_dir),
        report_to="none",

        # Required for vision SFT in Unsloth.
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        max_length=MAX_SEQ_LENGTH,

        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
    )

    collator = UnslothVisionDataCollator(model, processor)
    trainer_kwargs = {
        "model": model,
        "data_collator": collator,
        "train_dataset": train_ds,
        "args": args,
    }

    sig = inspect.signature(SFTTrainer.__init__)
    if "tokenizer" in sig.parameters:
        trainer_kwargs["tokenizer"] = processor
        print("SFTTrainer mode: Unsloth patched tokenizer=processor")
    else:
        _ensure_legacy_eos_token_if_needed(model, processor)
        trainer_kwargs["processing_class"] = processor.tokenizer
        print("SFTTrainer mode: TRL processing_class fallback")

    trainer = SFTTrainer(**trainer_kwargs)

    if torch.cuda.is_available():
        gpu_stats = torch.cuda.get_device_properties(0)
        start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
        max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
        print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
        print(f"Start reserved memory = {start_gpu_memory} GB.")

    trainer_stats = trainer.train()
    print(trainer_stats)

    adapter_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(adapter_dir)
    processor.save_pretrained(adapter_dir)

    write_json(run_dir / "train_config.json", {
        "config": cfg,
        "train_records": len(train_records),
        "trainer_stats": getattr(trainer_stats, "metrics", {}),
    })

    if torch.cuda.is_available():
        used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
        print(f"Peak reserved memory = {used_memory} GB.")

    print("saved adapter:", adapter_dir)

    del trainer, model, processor
    clear_cuda()
    return adapter_dir, getattr(trainer_stats, "metrics", None)

print("Helpers ready.")
print("SFTTrainer signature:", inspect.signature(SFTTrainer.__init__))


In [ ]:
# 7) Rule metrics + summary helpers

def answer_word_count(text):
    return len((text or "").split())

def is_one_paragraph(text):
    text = (text or "").strip()
    if not text:
        return False
    return len([p for p in re.split(r"\n\s*\n", text) if p.strip()]) <= 1

def has_markdown_bullets(text):
    return bool(re.search(r"(?m)^\s*([-*+]|\d+\.)\s+", text or ""))

def has_json_or_code_fence(text):
    return "```" in (text or "") or bool(re.search(r"^\s*[\{\[]", text or ""))

def format_pass(text, raw_had_think=False):
    text = strip_thinking(text or "")
    wc = answer_word_count(text)
    return (
        wc > 0
        and not raw_had_think
        and "<think" not in text.lower()
        and is_one_paragraph(text)
        and not has_json_or_code_fence(text)
        and wc <= 180
    )

def compute_rule_metrics(predictions):
    if not predictions:
        return {}

    answers = [p.get("generated_answer", "") for p in predictions]
    raw_had_think = [bool(p.get("had_think_raw", False)) for p in predictions]
    latencies = [p.get("latency_sec") for p in predictions if p.get("latency_sec") is not None]
    tokens = [p.get("output_tokens") for p in predictions if p.get("output_tokens") is not None]
    word_counts = [answer_word_count(strip_thinking(a)) for a in answers]

    return {
        "sample_count": len(predictions),
        "format_pass_rate": sum(format_pass(a, h) for a, h in zip(answers, raw_had_think)) / len(answers),
        "no_think_rate": sum("<think" not in (a or "").lower() for a in answers) / len(answers),
        "raw_no_think_rate": sum(not h for h in raw_had_think) / len(raw_had_think),
        "one_paragraph_rate": sum(is_one_paragraph(a) for a in answers) / len(answers),
        "markdown_bullet_rate": sum(has_markdown_bullets(a) for a in answers) / len(answers),
        "json_or_code_fence_rate": sum(has_json_or_code_fence(a) for a in answers) / len(answers),
        "empty_answer_rate": sum(wc == 0 for wc in word_counts) / len(word_counts),
        "answer_word_count_mean": sum(word_counts) / len(word_counts),
        "answer_word_count_p95": sorted(word_counts)[int(0.95 * (len(word_counts) - 1))],
        "latency_sec_mean": sum(latencies) / len(latencies) if latencies else None,
        "output_tokens_mean": sum(tokens) / len(tokens) if tokens else None,
    }

def mean(xs):
    xs = [x for x in xs if x is not None]
    return sum(xs) / len(xs) if xs else None

def summarize_judge(rows):
    if not rows:
        return {}
    return {
        "judge_sample_count": len(rows),
        "entailment_score_mean": mean([float(r.get("entailment_score")) for r in rows if r.get("entailment_score") is not None]),
        "factqa_precision_mean": mean([float(r.get("factqa_precision")) for r in rows if r.get("factqa_precision") is not None]),
        "factqa_recall_mean": mean([float(r.get("factqa_recall")) for r in rows if r.get("factqa_recall") is not None]),
        "clarity_mean": mean([float(r.get("clarity")) for r in rows if r.get("clarity") is not None]),
        "pedagogical_techniques_mean": mean([float(r.get("pedagogical_techniques")) for r in rows if r.get("pedagogical_techniques") is not None]),
        "critical_thinking_mean": mean([float(r.get("critical_thinking")) for r in rows if r.get("critical_thinking") is not None]),
        "format_adherence_mean": mean([float(r.get("format_adherence")) for r in rows if r.get("format_adherence") is not None]),
        "groundedness_mean": mean([float(r.get("groundedness")) for r in rows if r.get("groundedness") is not None]),
        "hallucination_rate": mean([1.0 if r.get("hallucination_flag") else 0.0 for r in rows if "hallucination_flag" in r]),
    }

def paired_win_rate(base_rows, ft_rows):
    base = {r["id"]: r for r in base_rows}
    ft = {r["id"]: r for r in ft_rows}
    win = tie = lose = 0
    by_split = defaultdict(lambda: {"win": 0, "tie": 0, "lose": 0})

    for row_id in sorted(set(base) & set(ft)):
        b, f = base[row_id], ft[row_id]

        def score(x):
            clarity = float(x.get("clarity", 0) or 0) / 5
            fmt = float(x.get("format_adherence", 0) or 0) / 5
            upt = float(x.get("pedagogical_techniques", 0) or 0) / 5
            ect = float(x.get("critical_thinking", 0) or 0) / 5
            fqa_p = float(x.get("factqa_precision", 0) or 0)
            grounded = float(x.get("groundedness", 0) or 0)
            halluc = 1.0 if x.get("hallucination_flag") else 0.0
            return 0.25*clarity + 0.20*fmt + 0.15*upt + 0.10*ect + 0.20*fqa_p + 0.10*grounded - 0.25*halluc

        bs, fs = score(b), score(f)
        if fs > bs + 1e-6:
            outcome = "win"; win += 1
        elif abs(fs - bs) <= 1e-6:
            outcome = "tie"; tie += 1
        else:
            outcome = "lose"; lose += 1
        by_split[f.get("split", "unknown")][outcome] += 1

    compared = win + tie + lose
    return {
        "overall": {
            "compared": compared,
            "win": win,
            "tie": tie,
            "lose": lose,
            "win_rate": win / compared if compared else None,
        },
        "by_split": dict(by_split),
    }

def build_experiment_summary(cfg, baseline_predictions, ft_predictions, baseline_judged, ft_judged):
    return {
        "config": cfg,
        "scope": "full_test = synthetic_test + real_world_test",
        "rule_metrics": {
            "baseline": compute_rule_metrics(baseline_predictions),
            "fine_tuned": compute_rule_metrics(ft_predictions),
        },
        "baseline_judge": summarize_judge(baseline_judged),
        "fine_tuned_judge": summarize_judge(ft_judged),
        "paired_win_rate": paired_win_rate(baseline_judged, ft_judged),
    }

def summary_to_dataframe(summary):
    flat = {}
    def put(prefix, d):
        for k, v in (d or {}).items():
            if isinstance(v, dict):
                put(f"{prefix}.{k}", v)
            else:
                flat[f"{prefix}.{k}"] = v

    put("baseline.rule", summary.get("rule_metrics", {}).get("baseline", {}))
    put("ft.rule", summary.get("rule_metrics", {}).get("fine_tuned", {}))
    put("baseline.judge", summary.get("baseline_judge", {}))
    put("ft.judge", summary.get("fine_tuned_judge", {}))

    suffixes = sorted({k.replace("baseline.", "").replace("ft.", "") for k in flat})
    rows = []
    for suffix in suffixes:
        b = flat.get("baseline." + suffix)
        f = flat.get("ft." + suffix)
        delta = (f - b) if isinstance(f, (int, float)) and isinstance(b, (int, float)) else None
        rows.append({"metric": suffix, "baseline": b, "fine_tuned": f, "delta": delta})
    return pd.DataFrame(rows)


In [ ]:
# 8) Gemini round-robin JSON judge with 3-key rate-limit failover
from google import genai
from google.genai import errors, types

JUDGE_SCHEMA = {
    "type": "OBJECT",
    "required": [
        "entailment_score",
        "factqa_precision",
        "factqa_recall",
        "clarity",
        "pedagogical_techniques",
        "critical_thinking",
        "format_adherence",
        "groundedness",
        "hallucination_flag",
        "rationale",
    ],
    "properties": {
        "entailment_score": {"type": "NUMBER"},
        "factqa_precision": {"type": "NUMBER"},
        "factqa_recall": {"type": "NUMBER"},
        "clarity": {"type": "INTEGER"},
        "pedagogical_techniques": {"type": "INTEGER"},
        "critical_thinking": {"type": "INTEGER"},
        "format_adherence": {"type": "INTEGER"},
        "groundedness": {"type": "NUMBER"},
        "hallucination_flag": {"type": "BOOLEAN"},
        "rationale": {"type": "STRING"},
    },
}

def get_thinking_level(name: str):
    name = str(name or "MINIMAL").upper()
    valid = {"MINIMAL", "LOW", "MEDIUM", "HIGH", "THINKING_LEVEL_UNSPECIFIED"}
    if name not in valid:
        raise ValueError(f"Unsupported GEMINI_THINKING_LEVEL={name!r}; use one of {sorted(valid)}")
    return getattr(types.ThinkingLevel, name)

def is_rate_limit_error(exc):
    code = getattr(exc, "code", None)
    message = str(getattr(exc, "message", "") or exc).lower()
    return code == 429 or "rate limit" in message or "quota" in message or "resource exhausted"

class GeminiRoundRobinJudge:
    def __init__(self, api_keys, model_name, rpm_limit_per_key=15, rpd_limit_per_key=500, cooldown_sec=60):
        self.clients = [genai.Client(api_key=k) for k in api_keys]
        self.model_name = model_name
        self.rpm_limit = rpm_limit_per_key
        self.rpd_limit = rpd_limit_per_key
        self.cooldown_sec = cooldown_sec
        self.next_idx = 0
        self.calls = [0 for _ in api_keys]
        self.rate_limit_errors = [0 for _ in api_keys]
        self.rate_limited_until = [0.0 for _ in api_keys]
        self.last_call = [0.0 for _ in api_keys]
        self.config = types.GenerateContentConfig(
            response_mime_type="application/json",
            response_json_schema=JUDGE_SCHEMA,
            temperature=0,
            max_output_tokens=1200,
            thinking_config=types.ThinkingConfig(
                thinking_level=get_thinking_level(GEMINI_THINKING_LEVEL)
            ),
        )

    def _take_slot(self):
        now = time.monotonic()
        for _ in range(len(self.clients)):
            idx = self.next_idx
            self.next_idx = (self.next_idx + 1) % len(self.clients)
            if self.calls[idx] >= self.rpd_limit:
                continue
            if self.rate_limited_until[idx] > now:
                continue
            min_gap = 60.0 / self.rpm_limit
            wait = max(0.0, min_gap - (now - self.last_call[idx]))
            if wait:
                time.sleep(wait)
            self.calls[idx] += 1
            self.last_call[idx] = time.monotonic()
            return idx, self.clients[idx]
        raise RuntimeError("No Gemini API key slot available; all keys are rate-limited or reached configured RPD.")

    def _mark_rate_limited(self, idx, exc):
        self.rate_limit_errors[idx] += 1
        self.rate_limited_until[idx] = time.monotonic() + self.cooldown_sec
        print(f"Gemini key slot {idx + 1} hit rate/quota limit; switching key. Error: {getattr(exc, 'message', exc)}")

    def judge(self, prompt, max_retries=5):
        if not self.clients:
            raise RuntimeError("RUN_GEMINI_JUDGE=True but GEMINI_API_KEYS is empty.")

        last_error = None
        for attempt in range(1, max_retries + 1):
            last_rate_limit = None
            for _ in range(len(self.clients)):
                idx, client = self._take_slot()
                try:
                    response = client.models.generate_content(
                        model=self.model_name,
                        contents=prompt,
                        config=self.config,
                    )
                    text = response.text or "{}"
                    try:
                        data = json.loads(text)
                    except json.JSONDecodeError:
                        match = re.search(r"\{.*\}", text, flags=re.S)
                        if not match:
                            raise
                        data = json.loads(match.group(0))
                    data["api_key_slot"] = idx + 1
                    data["retry_attempt"] = attempt
                    return data
                except errors.APIError as exc:
                    last_error = exc
                    if is_rate_limit_error(exc):
                        last_rate_limit = exc
                        self._mark_rate_limited(idx, exc)
                        continue
                    code = getattr(exc, "code", None)
                    message = str(getattr(exc, "message", "") or exc).lower()
                    retryable = code in {500, 502, 503, 504} or "unavailable" in message or "high demand" in message
                    if retryable:
                        wait = min(120, 10 * attempt)
                        print(
                            f"Gemini retryable error attempt {attempt}/{max_retries} "
                            f"on key slot {idx + 1}; sleeping {wait}s. Error: {getattr(exc, 'message', exc)}"
                        )
                        time.sleep(wait)
                        break
                    raise

            if last_rate_limit is not None and last_error is last_rate_limit:
                wait = min(120, 10 * attempt)
                print(f"All available Gemini keys rate-limited on attempt {attempt}/{max_retries}; sleeping {wait}s before retry.")
                time.sleep(wait)

        raise RuntimeError(f"Gemini judge failed after {max_retries} retries: {last_error}")

def build_judge_prompt(reference_row, prediction_row):
    generated = prediction_row.get("generated_answer", "")
    raw = prediction_row.get("raw_generated_answer", "")
    rule_format_pass = format_pass(generated, prediction_row.get("had_think_raw", False))

    return f"""You are evaluating a multimodal tutor answer for EduVidQA.

Use the transcript/frame context and reference answer as the judging basis. Judge only the generated final answer.
Return strict JSON with these fields:
- entailment_score: 0.0 to 1.0, whether generated answer is entailed by the reference/context.
- factqa_precision: 0.0 to 1.0, fraction of generated factual claims supported.
- factqa_recall: 0.0 to 1.0, fraction of important reference facts covered.
- clarity: integer 1 to 5, clear, concise, easy-to-understand educational answer.
- pedagogical_techniques: integer 1 to 5, use of tutoring techniques such as explanation, analogy, step-by-step simplification, or linking to visuals.
- critical_thinking: integer 1 to 5, whether the answer encourages conceptual understanding or reasoning rather than rote phrasing.
- format_adherence: integer 1 to 5, follows: one concise paragraph, no <think>, no hidden reasoning, no unnecessary markdown/JSON.
- groundedness: 0.0 to 1.0, support from transcript/context/reference.
- hallucination_flag: boolean, true if any unsupported or contradictory factual claim appears.
- rationale: one short sentence.

Important calibration:
- Clarity should generally be high for a good answer.
- Pedagogical techniques and critical thinking should be useful but not forced.
- Penalize unsupported claims even if wording is fluent.
- The rule-based format_pass for this answer is: {rule_format_pass}

Question/context:
{reference_row.get('text_input', '')}

Reference answer:
{reference_row.get('answer', '')}

Generated answer:
{generated}

Raw generated answer:
{raw}
"""

refs_by_id = {r["id"]: r for r in full_test_records}

judge = GeminiRoundRobinJudge(
    GEMINI_API_KEYS,
    GEMINI_MODEL,
    rpm_limit_per_key=GEMINI_RPM_LIMIT_PER_KEY,
    rpd_limit_per_key=GEMINI_RPD_LIMIT_PER_KEY,
    cooldown_sec=GEMINI_RATE_LIMIT_COOLDOWN_SEC,
) if RUN_GEMINI_JUDGE else None

def judge_predictions(predictions, out_path, label, force=False):
    if not RUN_GEMINI_JUDGE:
        print("RUN_GEMINI_JUDGE=False; skipping judge:", label)
        return []

    out_path = Path(out_path)
    rows = [] if force else read_jsonl_safe(out_path)
    done = {row["id"] for row in rows}

    if rows and len(done) >= len(predictions):
        print(f"SKIP judge {label}: {out_path} ({len(rows)} rows)")
        return rows

    for pred in tqdm(predictions, desc=f"Gemini judge {label}", unit="sample"):
        if pred["id"] in done:
            continue
        ref = refs_by_id[pred["id"]]
        scores = judge.judge(build_judge_prompt(ref, pred))
        row = {"id": pred["id"], "split": ref.get("split"), **scores}
        rows.append(row)
        done.add(pred["id"])
        write_jsonl(out_path, rows)

    print("saved", len(rows), out_path)
    return rows

print("Judge ready:", GEMINI_MODEL, "thinking", GEMINI_THINKING_LEVEL, "keys", len(GEMINI_API_KEYS))


In [ ]:
# 9) Full baseline generation + judge on full test
baseline_full_predictions = []
baseline_full_judged = []

if RUN_FINAL_FULL_EVAL:
    baseline_full_pred_path = BASELINE_DIR / "predictions_full.jsonl"
    baseline_full_judge_path = BASELINE_DIR / "judge_full.jsonl"

    if FORCE_RERUN or len(read_jsonl_safe(baseline_full_pred_path)) < len(full_test_records):
        base_model, base_processor = load_base_model()
        baseline_full_predictions = generate_predictions(
            base_model,
            base_processor,
            full_test_records,
            baseline_full_pred_path,
            "baseline_full",
            force=FORCE_RERUN,
        )
        del base_model, base_processor
        clear_cuda()
    else:
        baseline_full_predictions = read_jsonl_safe(baseline_full_pred_path)
        print("SKIP baseline full generation:", baseline_full_pred_path)

    baseline_full_judged = judge_predictions(
        baseline_full_predictions,
        baseline_full_judge_path,
        "baseline_full",
        force=FORCE_RERUN,
    )

print("baseline predictions/judged:", len(baseline_full_predictions), len(baseline_full_judged))
print("baseline rule metrics:", json.dumps(compute_rule_metrics(baseline_full_predictions), ensure_ascii=False, indent=2))


In [ ]:
# 10) Final full train: r16_e1 on all synthetic_train_clean
final_adapter_dir = None
final_train_metrics = None

if RUN_FINAL_FULL_TRAIN:
    final_run_name = f"final_fulltrain_{SELECTED_CONFIG['name']}"
    final_run_dir = FINAL_DIR / final_run_name

    final_adapter_dir, final_train_metrics = train_final_adapter(
        SELECTED_CONFIG,
        train_records_for_this_run,
        final_run_dir,
        force=FORCE_RERUN,
    )

    write_json(FINAL_DIR / "selected_final_config.json", {
        "selected_config": SELECTED_CONFIG,
        "final_run_name": final_run_name,
        "adapter_dir": str(final_adapter_dir),
        "train_records": len(train_records_for_this_run),
        "train_metrics": final_train_metrics,
        "note": "Final model trained on all synthetic_train_clean using r16_e1. Eval is full synthetic_test+real_world_test.",
    })

    print("Final adapter:", final_adapter_dir)
else:
    final_config_path = FINAL_DIR / "selected_final_config.json"
    assert final_config_path.exists(), "RUN_FINAL_FULL_TRAIN=False and no saved final config found."
    payload = read_json(final_config_path)
    final_adapter_dir = Path(payload["adapter_dir"])
    print("Loaded final adapter:", final_adapter_dir)


In [ ]:
# 11) Full fine-tuned generation + judge on full test
final_ft_full_predictions = []
final_ft_full_judged = []

if RUN_FINAL_FULL_EVAL:
    if final_adapter_dir is None:
        final_config_path = FINAL_DIR / "selected_final_config.json"
        assert final_config_path.exists(), "No final adapter. Run final full train first."
        payload = read_json(final_config_path)
        final_adapter_dir = Path(payload["adapter_dir"])

    final_run_name = f"final_fulltrain_{SELECTED_CONFIG['name']}"
    final_eval_dir = FINAL_DIR / final_run_name
    final_pred_path = final_eval_dir / "predictions_full.jsonl"
    final_judge_path = final_eval_dir / "judge_full.jsonl"

    if FORCE_RERUN or len(read_jsonl_safe(final_pred_path)) < len(full_test_records):
        model, processor = load_model_with_adapter(final_adapter_dir)
        final_ft_full_predictions = generate_predictions(
            model,
            processor,
            full_test_records,
            final_pred_path,
            f"{final_run_name}_full",
            force=FORCE_RERUN,
        )
        del model, processor
        clear_cuda()
    else:
        final_ft_full_predictions = read_jsonl_safe(final_pred_path)
        print("SKIP final full generation:", final_pred_path)

    final_ft_full_judged = judge_predictions(
        final_ft_full_predictions,
        final_judge_path,
        f"{final_run_name}_full",
        force=FORCE_RERUN,
    )

print("FT predictions/judged:", len(final_ft_full_predictions), len(final_ft_full_judged))
print("FT rule metrics:", json.dumps(compute_rule_metrics(final_ft_full_predictions), ensure_ascii=False, indent=2))


In [ ]:
# 12) Save final comparison summary + CSV report
final_summary = {}

if RUN_FINAL_FULL_EVAL and final_ft_full_predictions:
    final_summary = build_experiment_summary(
        SELECTED_CONFIG,
        baseline_full_predictions,
        final_ft_full_predictions,
        baseline_full_judged,
        final_ft_full_judged,
    )
    final_summary["config"].update({
        "base_model": BASE_MODEL,
        "judge_model": GEMINI_MODEL,
        "gemini_keys": len(GEMINI_API_KEYS),
        "gemini_thinking_level": GEMINI_THINKING_LEVEL,
        "rpm_per_key": GEMINI_RPM_LIMIT_PER_KEY,
        "rpd_per_key": GEMINI_RPD_LIMIT_PER_KEY,
        "rate_limit_errors_per_key": getattr(judge, "rate_limit_errors", []) if judge else [],
        "load_in_4bit": LOAD_IN_4BIT,
        "max_seq_length": MAX_SEQ_LENGTH,
        "gen_max_new_tokens": GEN_MAX_NEW_TOKENS,
        "gen_temperature": GEN_TEMPERATURE,
        "gen_top_p": GEN_TOP_P,
    })
    final_summary["paths"] = {
        "baseline_predictions_full": str(BASELINE_DIR / "predictions_full.jsonl"),
        "baseline_judge_full": str(BASELINE_DIR / "judge_full.jsonl"),
        "final_adapter": str(final_adapter_dir),
        "final_predictions_full": str(FINAL_DIR / f"final_fulltrain_{SELECTED_CONFIG['name']}" / "predictions_full.jsonl"),
        "final_judge_full": str(FINAL_DIR / f"final_fulltrain_{SELECTED_CONFIG['name']}" / "judge_full.jsonl"),
    }

    summary_path = FINAL_DIR / f"summary_full_{SELECTED_CONFIG['name']}.json"
    write_json(summary_path, final_summary)

    report_df = summary_to_dataframe(final_summary)
    report_csv = FINAL_DIR / f"comparison_full_{SELECTED_CONFIG['name']}.csv"
    report_df.to_csv(report_csv, index=False)

    print(json.dumps(final_summary, ensure_ascii=False, indent=2))
    print("saved summary:", summary_path)
    print("saved comparison csv:", report_csv)
    display(report_df)
else:
    print("No final full eval summary created. Check RUN_FINAL_FULL_EVAL and previous cell outputs.")

print("Gemini calls per key:", getattr(judge, "calls", []) if judge else [])
print("Gemini rate-limit errors per key:", getattr(judge, "rate_limit_errors", []) if judge else [])


## Run order

Run all cells top-to-bottom.

Expected flow:
1. Install + load dataset.
2. Generate baseline predictions on full test.
3. Judge baseline predictions with Gemini Round Robin.
4. Train `Qwen3.5-4B` LoRA `r16_e1` on full `synthetic_train_clean`.
5. Generate fine-tuned predictions on full test.
6. Judge fine-tuned predictions.
7. Save summary JSON + comparison CSV to Drive.

To change config later, edit `SELECTED_CONFIG` in cell 0 and set `FORCE_RERUN=True` only for the parts you intentionally want to overwrite.
